# Zarr Data Parsing for Neural Network Training

Opens CRIMAC-preprocessor zarr output, explores the data structure, and
extracts calibrated Sv arrays for echogram visualization, bottom detection,
and training patch generation.

**Key points:**
- The zarr stores `sv` as **linear** values (not dB)
- Bottom detection uses the original CRIMAC algorithms directly on the zarr
- dB conversion is only needed for echogram plotting
- Trailing all-NaN range bins are auto-trimmed for efficient plotting/patching

In [ ]:
import sys
sys.path.insert(0, '../..')

from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

# CRIMAC bottom detection (original code, works directly on zarr)
from raw_sensor_integration.rsi_bottomdetection import simple_bottom_detector, edge_bottom_detector
from raw_sensor_integration.rsi_bottomdetection.parameters import Parameters

In [ ]:
ZARR_PATH = Path.home() / 'Downloads' / 'crimac_output/test.zarr'

## 1. Open and Explore Zarr Structure

In [ ]:
ds = xr.open_zarr(ZARR_PATH, consolidated=False)
print(ds)

In [ ]:
# Dimensions and coordinates
print(f"Sv dimensions:   {ds.sv.dims}")
print(f"Sv shape:        {ds.sv.shape}")
print("")
print(f"Frequencies (Hz): {ds.frequency.values}")
print(f"Ping time range:  {pd.Timestamp(ds.ping_time.values[0])} → {pd.Timestamp(ds.ping_time.values[-1])}")
print(f"Total pings:      {ds.dims['ping_time']}")
print(f"Range:            0 – {ds.range.values[-1]:.1f} m  ({ds.dims['range']} samples)")
print("")
print(f"Available variables: {list(ds.data_vars)}")
print("")
# Source files covered
raw_files = np.unique(ds.raw_file.values)
print(f"Source .raw files ({len(raw_files)}):")
for f in raw_files:
    print(f"  {f}")

In [ ]:
# Pulse length and channel IDs
print("Pulse lengths per frequency:")
for freq, ch, pl in zip(ds.frequency.values, ds.channel_id.values, ds.pulse_length.values):
    print(f"  {freq/1000:.1f} kHz  channel={ch[:40]}  pulse={pl*1000:.4f} ms")

print(f"\nHeave range: {float(ds.heave.min()):.3f} – {float(ds.heave.max()):.3f} m")
print(f"Speed range: {float(ds.speed.min()):.1f} – {float(ds.speed.max()):.1f} knots")

## 2. Helper: Load Sv in dB for Plotting

The zarr stores `sv` as **linear** values. The CRIMAC bottom detectors work
directly on linear sv (no conversion needed). But for echogram plotting we
need dB. This adapter converts and auto-trims trailing NaN rows.

In [ ]:
from raw_sensor_integration.preprocessing import load_sv_from_zarr

In [ ]:
# Quick sanity check
sv_db, range_array, ping_times, pulse_duration, heave, freq_used = \
    load_sv_from_zarr(ZARR_PATH, frequency_hz=38000)

print(f"Frequency used:  {freq_used/1000:.1f} kHz")
print(f"sv_db shape:     {sv_db.shape}  (n_range × n_pings)")
print(f"range_array:     {range_array[0]:.2f} – {range_array[-1]:.1f} m")
print(f"ping_times:      {ping_times[0]} → {ping_times[-1]}")
print(f"pulse_duration:  {pulse_duration*1000:.4f} ms")
print(f"heave range:     {heave.min():.3f} – {heave.max():.3f} m")
print("")
print(f"Sv range: {np.nanmin(sv_db):.1f} – {np.nanmax(sv_db):.1f} dB re 1 m⁻¹")
print("  (calibrated Sv: typical water column ≈ -80 to -34 dB re 1 m⁻¹)")

## 3. Echogram Visualization

`sv_db` is now the same shape `(n_range, n_pings)` as in notebooks 03/05 — same plotting code works.

In [ ]:
from raw_sensor_integration.visualization import plot_echogram

In [ ]:
# Auto-range echogram
vmin_auto, vmax_auto = np.nanpercentile(sv_db, [2, 98])
print(f"Auto range: {vmin_auto:.1f} to {vmax_auto:.1f} dB re 1 m⁻¹")
plot_echogram(sv_db, '38 kHz Sv — auto range', vmin=vmin_auto, vmax=vmax_auto,
              range_array=range_array)
plt.show()

In [ ]:
# Fixed range (standard fisheries Sv colourscale)
plot_echogram(sv_db, '38 kHz Sv — fixed scale [-80, -34] dB', vmin=-80, vmax=-34,
              range_array=range_array)
plt.show()

In [ ]:
# Both frequencies side by side
freqs = ds.frequency.values
n_freq = len(freqs)

fig, axes = plt.subplots(1, n_freq, figsize=(7 * n_freq, 6), sharey=True)
if n_freq == 1:
    axes = [axes]

for ax, f in zip(axes, freqs):
    sv_f, range_f, _, _, _, _ = load_sv_from_zarr(ZARR_PATH, frequency_hz=f)
    v0, v1 = np.nanpercentile(sv_f, [2, 98])
    extent_f = [0, sv_f.shape[1], range_f[-1], range_f[0]]
    im = ax.imshow(sv_f, aspect='auto', cmap='viridis', vmin=v0, vmax=v1,
                   origin='upper', extent=extent_f)
    ax.set_title(f'{f/1000:.0f} kHz  {sv_f.shape[0]}×{sv_f.shape[1]}')
    ax.set_xlabel('Ping')
    ax.set_ylabel('Range (m)')
    plt.colorbar(im, ax=ax, shrink=0.8, label='Sv (dB re 1 m⁻¹)')

plt.suptitle('All frequencies', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Bottom Detection (CRIMAC algorithms)

Using the original CRIMAC bottom detection code directly on the zarr.
These algorithms operate on **linear sv** — no dB conversion needed.

- **Simple**: threshold + back-step along leading edge
- **Edge**: Gaussian-derivative convolution (more accurate for flat bottom)

In [ ]:
# Run CRIMAC bottom detection directly on the zarr Dataset
params = Parameters(minimum_range=10.0, offset=0.5, threshold_log_sv=-31.0)

b_simple = simple_bottom_detector.detect_bottom(ds, params)
b_edge = edge_bottom_detector.detect_bottom(ds, params)

# Ensemble: per-ping median of both detectors
b_stack = np.stack([b_simple.values, b_edge.values])
b_ensemble = np.nanmedian(b_stack, axis=0)

n_samples, n_pings = sv_db.shape

print(f"{'Detector':<12} {'Valid':>8} {'Median':>8} {'Min':>8} {'Max':>8}")
print("-" * 48)
for name, arr in [('Simple', b_simple.values), ('Edge', b_edge.values), ('Ensemble', b_ensemble)]:
    v = np.mean(~np.isnan(arr))
    print(f"{name:<12} {v:>7.1%} {np.nanmedian(arr):>7.1f}m {np.nanmin(arr):>7.1f}m {np.nanmax(arr):>7.1f}m")

In [ ]:
# Echogram with bottom overlay
vmin_v, vmax_v = np.nanpercentile(sv_db, [2, 98])

fig, ax = plt.subplots(figsize=(14, 6))
ax.imshow(sv_db, aspect='auto', cmap='viridis', vmin=vmin_v, vmax=vmax_v,
          origin='upper', extent=[0, n_pings, range_array[-1], range_array[0]])

# CRIMAC returns absolute depth (heave + draft + range - offset).
# Our echogram y-axis is range from transducer, so subtract heave+draft.
heave_draft = (ds['heave'] + ds['transducer_draft'][0]).values
ping_x = np.arange(len(b_ensemble))

for name, arr, color in [('Simple', b_simple.values, 'cyan'),
                          ('Edge', b_edge.values, 'yellow'),
                          ('Ensemble', b_ensemble, 'red')]:
    # Convert absolute depth back to range-from-transducer for plotting
    range_vals = arr - heave_draft
    valid = ~np.isnan(range_vals)
    ax.plot(ping_x[valid], range_vals[valid], color=color, linewidth=1.0, label=name, alpha=0.8)

ax.set_xlabel('Ping number')
ax.set_ylabel('Range (m)')
ax.set_title('38 kHz Sv with CRIMAC bottom detection')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

In [ ]:
from raw_sensor_integration.bottom_detection import bottom_to_sample_index, create_bottom_mask

# Binary segmentation masks — U-Net targets
# Convert CRIMAC absolute depth → range from transducer for mask creation
heave_draft = (ds['heave'] + ds['transducer_draft'][0]).values
b_range = b_ensemble - heave_draft  # range from transducer (meters)

# Convert range (meters) → sample index using existing module function
bottom_indices = bottom_to_sample_index(b_range, range_array)

# Create masks using existing module functions
mask_below = create_bottom_mask(n_samples, n_pings, bottom_indices, mode='below')
mask_line = create_bottom_mask(n_samples, n_pings, bottom_indices, mode='line', line_thickness=3)

labeled = np.sum(bottom_indices >= 0)
print(f"Labeled pings: {labeled}/{n_pings} ({labeled/n_pings:.1%})")

extent_m = [0, n_pings, range_array[-1], range_array[0]]
vmin_v, vmax_v = np.nanpercentile(sv_db, [2, 98])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].imshow(sv_db, aspect='auto', cmap='viridis', vmin=vmin_v, vmax=vmax_v,
               origin='upper', extent=extent_m)
valid = ~np.isnan(b_range)
ping_x = np.arange(n_pings)
axes[0].plot(ping_x[valid], b_range[valid], color='red', linewidth=1.0)
axes[0].set_title('Sv + detection')

axes[1].imshow(mask_line, aspect='auto', cmap='gray_r', vmin=0, vmax=1,
               origin='upper', extent=extent_m)
axes[1].set_title('Line mask (3 px)')

axes[2].imshow(mask_below, aspect='auto', cmap='gray_r', vmin=0, vmax=1,
               origin='upper', extent=extent_m)
axes[2].set_title('Below mask (U-Net target)')

for ax in axes:
    ax.set_xlabel('Ping')
    ax.set_ylabel('Range (m)')
plt.tight_layout()
plt.show()

## 5. Training Patch Extraction

Extract fixed-width patches as numpy arrays for PyTorch.
Single frequency gives `(1, n_range, patch_width)`, multi-frequency stacks
on axis 0 to give `(n_freq, n_range, patch_width)`.

In [ ]:
from raw_sensor_integration.preprocessing import (
    extract_patches, extract_multifreq_patches
)

In [ ]:
# Count available patches
PATCH_WIDTH = 256
STRIDE = 128

patches = list(extract_patches(sv_db, mask_below,
                                patch_width=PATCH_WIDTH, stride=STRIDE,
                                require_bottom=True))

print(f"Patches extracted: {len(patches)}")
print(f"Image shape: {patches[0][0].shape}  (C, H, W)")
print(f"Label shape: {patches[0][1].shape}")
print(f"Image dtype: {patches[0][0].dtype}")
print(f"Image range: [{patches[0][0].min():.3f}, {patches[0][0].max():.3f}]  (0–1 normalised)")

In [ ]:
# Preview first few patches
n_preview = min(4, len(patches))
fig, axes = plt.subplots(2, n_preview, figsize=(5 * n_preview, 8))

for i in range(n_preview):
    img, lbl = patches[i]
    axes[0, i].imshow(img[0], aspect='auto', cmap='viridis', vmin=0, vmax=1, origin='upper')
    axes[0, i].set_title(f'Patch {i} — Sv')
    axes[0, i].set_xlabel('Ping')
    axes[0, i].set_ylabel('Range sample')

    axes[1, i].imshow(lbl, aspect='auto', cmap='gray_r', vmin=0, vmax=1, origin='upper')
    axes[1, i].set_title(f'Patch {i} — mask')
    axes[1, i].set_xlabel('Ping')
    axes[1, i].set_ylabel('Range sample')

plt.suptitle(f'Training patches  ({PATCH_WIDTH} pings wide, stride {STRIDE})')
plt.tight_layout()
plt.show()

In [ ]:
# Multi-frequency patches
mf_patches = list(extract_multifreq_patches(
    ZARR_PATH,
    frequencies_hz=list(ds.frequency.values),
    mask=mask_below,
    patch_width=PATCH_WIDTH,
    stride=STRIDE,
))

print(f"Multi-freq patches: {len(mf_patches)}")
print(f"Image shape: {mf_patches[0][0].shape}  (n_freq, H, W)")

# Preview: one patch, all frequencies
img_mf, lbl_mf = mf_patches[0]
n_freq_actual = img_mf.shape[0]
fig, axes = plt.subplots(1, n_freq_actual + 1, figsize=(6 * (n_freq_actual + 1), 5))
for fi in range(n_freq_actual):
    axes[fi].imshow(img_mf[fi], aspect='auto', cmap='viridis', vmin=0, vmax=1)
    axes[fi].set_title(f'{ds.frequency.values[fi]/1000:.0f} kHz')
    axes[fi].set_xlabel('Ping')
axes[-1].imshow(lbl_mf, aspect='auto', cmap='gray_r', vmin=0, vmax=1)
axes[-1].set_title('Bottom mask')
axes[-1].set_xlabel('Ping')
plt.suptitle('Multi-frequency patch (first patch)')
plt.tight_layout()
plt.show()

## 6. Quick PyTorch Check

Verify patches can be converted to `torch.Tensor` (no install required if torch is absent — cell is guarded).

In [ ]:
try:
    import torch
    img_t, lbl_t = patches[0]
    t_img = torch.from_numpy(img_t)           # (1, H, W)
    t_lbl = torch.from_numpy(lbl_t.astype(np.float32))  # (H, W)
    print(f"torch.Tensor image: {t_img.shape}  dtype={t_img.dtype}")
    print(f"torch.Tensor label: {t_lbl.shape}  dtype={t_lbl.dtype}")
    print("Ready to feed into a DataLoader.")
except ImportError:
    print("torch not installed — patches are plain numpy arrays, castable via torch.from_numpy()")

## 7. Running the Preprocessor on a GPU Cluster

The preprocessing pipeline:
1. Run the CRIMAC Docker preprocessor on raw files to produce `_sv.zarr`
2. Use this notebook's code to build training patches from the zarr

On HPC clusters where Docker is unavailable, use Singularity:

```bash
singularity pull crimac_preprocessor.sif docker://crimac/preprocessor

# Step 1 — parquet index
singularity run \
  --bind /path/to/raw:/datain,/path/to/output:/dataout,/tmp:/workin \
  --env OUTPUT_TYPE=parquet,MAIN_FREQ=38000,MAX_RANGE_SRC=500,OUTPUT_NAME=survey2024 \
  crimac_preprocessor.sif

# Step 2 — zarr
singularity run \
  --bind /path/to/raw:/datain,/path/to/output:/dataout,/tmp:/workin \
  --env OUTPUT_TYPE=zarr,MAIN_FREQ=38000,MAX_RANGE_SRC=500,OUTPUT_NAME=survey2024 \
  crimac_preprocessor.sif
```

If neither Docker nor Singularity is available, install the dependencies and run directly:

```bash
pip install pyecholab2 xarray zarr numcodecs rechunker dask pandas pyarrow scipy
pip install git+https://github.com/CRIMAC-WP4-Machine-learning/pyEcholab.git

OUTPUT_TYPE=zarr MAIN_FREQ=38000 MAX_RANGE_SRC=500 OUTPUT_NAME=survey2024 \
  python CRIMAC_preprocess.py
```

Preprocessing is CPU-only (~2-6 hours for ~50 GB of raw files). GPU is only needed for training.

## 8. Processing the 2024 Oslofjord Survey Files

The `test.zarr` above was created from a 2021 file (D20210507-T154515.raw, 5 frequencies, FM broadband).
The 2024 Oslofjord files use CW mode (38 kHz + 200 kHz) and are the primary dataset for notebooks 01–05.

Running the CRIMAC preprocessor on those files produces a zarr for comparison with the raw-based bottom detection in notebook 05.

In [ ]:

RAW_2024_DIR = str(Path.home() / 'Downloads' / 'drive-download-20251116T003920Z-1-001')
CRIMAC_OUTPUT = str(Path.home() / 'Downloads' / 'crimac_output')
OUTPUT_NAME = "survey2024"

# Step 1: generate ping-distance parquet (required before zarr creation)
cmd_parquet = (
    f"docker run --rm "
    f"-v {RAW_2024_DIR}:/datain:ro "
    f"-v {CRIMAC_OUTPUT}:/dataout "
    f"-v /tmp:/workin "
    f"-e OUTPUT_TYPE=parquet "
    f"-e MAIN_FREQ=38000 "
    f"-e MAX_RANGE_SRC=500 "
    f"-e OUTPUT_NAME={OUTPUT_NAME} "
    f"crimac/preprocessor"
)

# Step 2: create the zarr
cmd_zarr = cmd_parquet.replace("OUTPUT_TYPE=parquet", "OUTPUT_TYPE=zarr")

print("Step 1 — parquet:")
print(cmd_parquet)
print()
print("Step 2 — zarr:")
print(cmd_zarr)
print()
print("Run these in a terminal. Each step takes a few minutes on ~5 raw files.")
print(f"Output zarr will be at: {CRIMAC_OUTPUT}/{OUTPUT_NAME}.zarr")